In [4]:
from pathlib import Path
import pandas as pd
import os
import sys

In [5]:
os.getcwd()

'c:\\Users\\Harry Turner\\Documents\\socialMediaProjects\\2026\\social-stories\\christmas-songs'

In [6]:
DATA_DIR = Path("data")

global_combined = pd.concat(
    [pd.read_csv(DATA_DIR / "spotify_daily_global.csv", dtype={
    "region": str, "previous_rank": int, "peak_rank": int, "days_on_chart": int, "streams": int,
    "track_name": str, "artists": str, "track_uri": str, "labels": str}, parse_dates=["date", "release_date"])
    ]
    )

In [7]:
global_combined.head(20)

,date,region,rank,previous_rank,peak_rank,days_on_chart,streams,track_name,artists,track_uri,labels,release_date
0,2020-10-01,global,1,1,1,53,5681338,Mood (feat. iann dior),"24kGoldn, iann dior",spotify:track:3tjFYV6RSFtuktYl3ZtYcq,Records/Columbia,2021-03-26
1,2020-10-01,global,2,2,1,56,4786211,WAP (feat. Megan Thee Stallion),"Cardi B, Megan Thee Stallion",spotify:track:4Oun2ylbjFKMPTiaSbbCih,Atlantic/KSR,2020-08-07
2,2020-10-01,global,3,3,3,49,4502404,Lemonade (feat. NAV),"Internet Money, Gunna, Don Toliver, NAV",spotify:track:7hxHWCCAIIxFLCzvDgnQHX,Internet Money Records / TenThousand Projects,2020-08-28
3,2020-10-01,global,4,4,2,63,4066369,Hawái,Maluma,spotify:track:1yoMvmasuxZfqHEipJhRbp,Sony Music Latin,2020-08-21
4,2020-10-01,global,5,5,1,42,4016787,Dynamite,BTS,spotify:track:0t1kP63rueHleOhQkYSXFY,BIGHIT MUSIC,2020-11-20
5,2020-10-01,global,6,6,4,14,3441822,Holy (feat. Chance The Rapper),"Justin Bieber, Chance the Rapper",spotify:track:5u1n1kITHCxxp8twBcZxWy,RBMG/Def Jam,2020-09-18
6,2020-10-01,global,7,7,1,308,3410272,Blinding Lights,The Weeknd,spotify:track:0VjIjW4GlUZAMYd2vXMi3b,Republic Records,2020-03-20
7,2020-10-01,global,8,8,4,320,3321198,Watermelon Sugar,Harry Styles,spotify:track:6UelLqGlWMcVH1E5c4H7lY,Columbia,2019-12-13
8,2020-10-01,global,9,9,9,91,3308897,For The Night,"Pop Smoke, Lil Baby, DaBaby",spotify:track:0PvFJmanyNQMseIFrU708S,Victor Victor Worldwide,2020-07-03
9,2020-10-01,global,10,10,1,112,2921831,Savage Love (Laxed - Siren Beat),"Jawsh 685, Jason Derulo",spotify:track:1xQ6trAsedVPCdbtDAmk0c,Columbia,2020-06-11


## Calendar heatmaps: Christmas on Spotify's global top 100

Two charts, each with one row per season from 25 October to 7 January and Christmas Day outlined:

1. **All I Want for Christmas Is You**, coloured by its position in the top 100, with the number of days it spent there each season on the right.
2. **Christmas songs in the top 100**, coloured by how many Christmas songs were in the top 100 that day, with each season's peak on the right.

In [8]:
import numpy as np
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Rectangle

# bwcharts isn't installed in this venv, so load it straight from its folder
BWCHARTS_DIR = r"C:\Users\Harry Turner\Documents\bwcharts"
if BWCHARTS_DIR not in sys.path:
    sys.path.insert(0, BWCHARTS_DIR)
import bwcharts as bw

# Mariah Carey's original only (matching on the URI leaves out the Justin Bieber duet and covers)
MARIAH_URI = "spotify:track:0bYg9bo50gSsH3LtXe2SQn"
mariah_rank = global_combined.loc[global_combined["track_uri"] == MARIAH_URI].set_index("date")["rank"]

# one row per chart day: the season it belongs to (named by its October year) and days since 1 Oct
cal = pd.DataFrame({"date": pd.DatetimeIndex(sorted(global_combined["date"].unique()))})
cal["season"] = np.where(cal["date"].dt.month >= 10, cal["date"].dt.year, cal["date"].dt.year - 1)
cal["day"] = (cal["date"] - pd.to_datetime(cal["season"].astype(str) + "-10-01")).dt.days
cal["rank"] = cal["date"].map(mariah_rank)          # NaN = not in the top 200 that day

rank_grid = cal.pivot(index="season", columns="day", values="rank")
has_chart = cal.pivot(index="season", columns="day", values="date").notna()   # False past 28 Feb

# check the story: first and last day on the chart, days at No. 1, and Christmas Day each season
on_chart = cal.dropna(subset=["rank"])
on_chart.groupby("season").agg(
    first_day=("date", "min"), last_day=("date", "max"), days_on_chart=("rank", "size"),
    days_at_no1=("rank", lambda r: (r == 1).sum()),
).join(on_chart[on_chart["date"].dt.strftime("%m-%d") == "12-25"].set_index("season")["rank"]
       .rename("christmas_day_rank"))

,first_day,last_day,days_on_chart,days_at_no1,christmas_day_rank
season,,,,,
2020,2020-11-01,2020-12-31,61,7,1.0
2021,2021-11-01,2021-12-31,61,12,1.0
2022,2022-11-01,2022-12-31,61,16,1.0
2023,2023-11-01,2023-12-31,61,27,1.0
2024,2024-11-01,2024-12-31,61,5,1.0
2025,2025-11-01,2025-12-31,60,7,1.0


In [ ]:
# Shared drawing code for the calendar heatmaps
CUTOFF = 100

# both charts show 25 Oct to 7 Jan: Christmas songs only reach the top 100 in November and
# December, and a week either side is enough to see them switch on and off
SEASON_START = pd.Timestamp("2020-10-01")             # any season works; only the day offsets matter
day_from = (pd.Timestamp("2020-10-25") - SEASON_START).days
day_to = (pd.Timestamp("2021-01-07") - SEASON_START).days
seasons = list(rank_grid.index)

# five bands in one hue, brighter = more; the top band is lifted towards white so it stands out
BAND_COLOURS = [bw._mix(bw.PRIMARY, "#FFFFFF", 0.25)] + [bw._mix(bw.BACKGROUND, bw.PRIMARY, t)
                                                         for t in (0.85, 0.66, 0.47, 0.3)]
OFF_CHART = bw._mix(bw.BACKGROUND, bw.NEUTRAL, 0.35)


def to_bands(grid, edges):
    """Band index for each value of a season x day grid (window only). edges are [low, high)
    and run from the top band to the bottom one; anything else on a chart day is the off band."""
    values = grid.loc[:, day_from:day_to].to_numpy(dtype=float)
    charted = has_chart.loc[:, day_from:day_to].to_numpy()
    bands = np.full(values.shape, np.nan)
    for i, (lo, hi) in enumerate(edges):
        bands[(values >= lo) & (values < hi)] = i
    bands[np.isnan(bands) & charted] = len(edges)
    return bands


def calendar_heatmap(bands, labels, off_label, *, title, subtitle, standfirst, path,
                     side_values=None, side_header=None, note=None):
    """One row per season, one cell per day, Christmas Day outlined. side_values (one per
    season) are written in a column on the right under side_header."""
    fig, ax = bw.chart(
        title, subtitle, standfirst=standfirst, note=note,
        source="Source: Spotify Charts, global daily top 200, 1 October 2020 to 28 February 2026",
        size="landscape", grid=None,
    )

    n_rows = bands.shape[0]
    cmap = ListedColormap(BAND_COLOURS[:len(labels)] + [OFF_CHART])
    norm = BoundaryNorm(np.arange(-0.5, len(labels) + 1), cmap.N)
    # cell edges in days since 1 Oct, so dates on the x axis keep their real positions
    ax.pcolormesh(np.arange(day_from, day_to + 2), np.arange(n_rows + 1),
                  np.ma.masked_invalid(bands), cmap=cmap, norm=norm,
                  edgecolors=bw.BACKGROUND, linewidth=0.2)
    for y in range(1, n_rows):                           # a clear gap between seasons
        ax.axhline(y, color=bw.BACKGROUND, linewidth=2.5, zorder=2)

    # seasons down the side (first at the top), dates along the bottom
    ax.set_xlim(day_from, day_to + 1)
    ax.set_ylim(n_rows, 0)
    ax.set_yticks(np.arange(n_rows) + 0.5, [f"{s}/{str(s + 1)[2:]}" for s in seasons])
    tick_dates = pd.to_datetime(["2020-11-01", "2020-12-01", "2021-01-01"])
    ax.set_xticks([(d - SEASON_START).days for d in tick_dates],
                  [f"{d.day} {d:%b}" for d in tick_dates])
    for t in ax.get_xticklabels():
        t.set_horizontalalignment("left")
    ax.tick_params(axis="x", pad=3)

    # Christmas Day: a white outline round the 25 December column, labelled above
    xmas = (pd.Timestamp("2020-12-25") - SEASON_START).days
    ax.add_patch(Rectangle((xmas, 0), 1, n_rows, fill=False, edgecolor=bw.TEXT, linewidth=1.1,
                           zorder=3, clip_on=False))
    ax.annotate("Christmas Day", xy=(xmas + 0.5, 0), xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=7, fontweight="semibold", color=bw.TEXT,
                annotation_clip=False)

    # summary column on the right, one number per season
    if side_values is not None:
        ax.annotate(side_header, xy=(1, 0), xycoords=("axes fraction", "data"), xytext=(12, 4),
                    textcoords="offset points", ha="center", va="bottom", fontsize=7,
                    fontweight="semibold", color=bw.TEXT, annotation_clip=False,
                    multialignment="center")
        for row, value in enumerate(side_values):
            ax.annotate(f"{value:.0f}", xy=(1, row + 0.5), xycoords=("axes fraction", "data"),
                        xytext=(12, 0), textcoords="offset points", ha="center",
                        va="center_baseline", fontsize=8, fontweight="semibold", color=bw.TEXT,
                        annotation_clip=False)

    bw.key(ax, dict(zip(labels + [off_label], BAND_COLOURS[:len(labels)] + [OFF_CHART])), dy=18)

    Path(path).parent.mkdir(exist_ok=True)
    bw.save(fig, path)
    return fig

In [ ]:
# Heatmap 1: All I Want for Christmas Is You, position in the top 100
RANK_EDGES = [(1, 2), (2, 6), (6, 11), (11, 51), (51, CUTOFF + 1)]
RANK_LABELS = ["No. 1", "2 to 5", "6 to 10", "11 to 50", "51 to 100"]

# days in the top 100 each season (the whole season, which is the same as the window)
days_in_top_100 = (rank_grid <= CUTOFF).sum(axis=1)

# check the story: first and last day in the top 100, and days in it, each season
in_top = cal[cal["rank"] <= CUTOFF]
print(in_top.groupby("season").agg(first_day=("date", "min"), last_day=("date", "max"),
                                   days_in_top_100=("rank", "size")))

calendar_heatmap(
    to_bands(rank_grid, RANK_EDGES), RANK_LABELS, f"Not in top {CUTOFF}",
    # a non-breaking space keeps "1 November" on one line
    title="In 2023 and 2025, Mariah held the top 100 from *1 November*",
    subtitle="Position of All I Want for Christmas Is You on Spotify's global daily chart, "
             "top 100 only, 25 October to 7 January",
    standfirst="In 2020 and 2021 the song didn't settle in the top 100 until the second week of "
               "November. Every season it drops out on 30 December, then returns for New Year's Eve.",
    path="charts/aiwfc_calendar_heatmap_top100.png",
    side_values=days_in_top_100, side_header="Days in\ntop 100",
)

In [ ]:
# Heatmap 2: how many Christmas songs are in the top 100 each day
#
# The data has no genre, so a Christmas song is one with a Christmas word in its title, less a
# few false matches, plus a few seasonal songs whose titles don't say Christmas. Both lists were
# checked by hand against every track that reached the top 100 between 25 Oct and 7 Jan.
XMAS_WORDS = (
    r"christmas|xmas|santa|jingle|navidad|rudolph|sleigh|mistletoe|reindeer|silent night|"
    r"holy night|merry|frosty|joy to the world|deck the hall|little drummer|carol of|grinch|"
    r"wonderful time|let it snow|snowman|winter wonderland|feliz|holiday|o come|noel|tannenbaum|"
    r"do you hear what|saint nick|fairytale of new york|underneath the tree|my only wish|"
    r"mele kalikimaka|cold outside|one more sleep|naughty list|2000 miles|silver bells|"
    r"my kind of present"
)
# titles that match a word above but aren't Christmas songs
# (Feid, Rvssian's Santa, Drake's 7969 Santa, Lil Nas X, Stray Kids, Kesha's Holiday Road)
NOT_XMAS = r"^feliz cumpleaños|^santa$|^7969 santa|^holiday$|^holiday road"
# Christmas songs whose titles give no clue
XMAS_EXTRA = {("Hallelujah", "Pentatonix"), ("I'll Be Home", "Meghan Trainor")}

titles = global_combined["track_name"].fillna("")
is_xmas = (
    (titles.str.contains(XMAS_WORDS, case=False) & ~titles.str.contains(NOT_XMAS, case=False))
    | pd.Series(list(zip(titles, global_combined["artists"])), index=global_combined.index)
        .isin(XMAS_EXTRA)
)
top_100 = global_combined[global_combined["rank"] <= CUTOFF]
xmas_per_day = top_100[is_xmas.loc[top_100.index]].groupby("date").size()

cal["xmas_songs"] = cal["date"].map(xmas_per_day).fillna(0)
xmas_grid = cal.pivot(index="season", columns="day", values="xmas_songs")

# days with at least one Christmas song in the top 100, counted over the days the chart shows so
# the number matches its row (2020/21 runs on past the window: Sia's Snowman lasted to 28 Feb)
days_with_xmas = (xmas_grid.loc[:, day_from:day_to] > 0).sum(axis=1)

# check the story: days with a Christmas song (shown and whole season), and each season's peak
print(f"{top_100.loc[is_xmas.loc[top_100.index], 'track_uri'].nunique()} different Christmas "
      "tracks reached the top 100")
print(pd.DataFrame({"days_shown": days_with_xmas, "days_whole_season": (xmas_grid > 0).sum(axis=1),
                    "peak": xmas_grid.max(axis=1), "peak_day": xmas_grid.idxmax(axis=1)
                    .map(lambda d: (SEASON_START + pd.Timedelta(days=int(d))).strftime("%d %b"))}))

XMAS_EDGES = [(75, CUTOFF + 1), (50, 75), (25, 50), (10, 25), (1, 10)]
XMAS_LABELS = ["75 or more", "50 to 74", "25 to 49", "10 to 24", "1 to 9"]

calendar_heatmap(
    to_bands(xmas_grid, XMAS_EDGES), XMAS_LABELS, "None",
    title="On Christmas Day, Christmas songs fill *more than three-quarters* of the top 100",
    subtitle="Christmas songs in Spotify's global daily top 100, 25 October to 7 January",
    standfirst="The count peaks on 25 December every season. In 2025 it reached 86, "
               "the most since the data starts in 2020.",
    path="charts/xmas_songs_calendar_heatmap_top100.png",
    side_values=days_with_xmas, side_header="Days with\nChristmas songs",
    note="Note: Days are counted from 25 October to 7 January. In 2020/21, Sia's Snowman "
         "stayed in the top 100 until 28 February.",
)